# Homework: your session 6 answers, in pandas, worked solution

Every answer here is pinned with an `assert` to the number in the session 6
worked answers, so the two languages are checked against each other.

In [ ]:
import os
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

plays = pd.read_csv("data/clean/plays.csv")
artists = pd.read_csv("data/clean/artists.csv")

print(plays.shape, artists.shape)

## 1. How many artists?

In [ ]:
assert len(artists) == 40
print(len(artists), "artists")

`len()` on a DataFrame counts the rows, exactly as it did on the list of
dictionaries. `artists.shape[0]` is the same number.

## 2. Artists from the Netherlands

In [ ]:
dutch = artists[artists["country"] == "Netherlands"][["artist_name", "genre"]]
dutch = dutch.sort_values("artist_name")

assert list(dutch["artist_name"]) == ["DJ Kompas", "Neon Harbour", "The Long Commute"]
dutch

Filter the rows, then pick the columns, then sort. Three steps, in the order
they happen.

## 3. Artists with an unknown `formed_year`

In [ ]:
artists.dtypes

`formed_year` is `float64`, a decimal, even though every year is a whole
number. A missing value in pandas is `NaN`, and `NaN` is a special decimal
value, so any number column with a gap in it is stored as decimals. When you
see `2014.0` where you expected `2014`, look for missing values.

Now the trap. The SQL lesson was that `= NULL` finds nothing. The pandas
equivalent fails in the same silent way:

In [ ]:
print(len(artists[artists["formed_year"] == None]), "rows found with == None")   # noqa: E711

Zero, and no error. A missing value is not equal to anything, not even
`None`. The only way to ask is `isna()`, which is `IS NULL`:

In [ ]:
unknown = artists[artists["formed_year"].isna()][["artist_name", "country", "genre"]]

assert set(unknown["artist_name"]) == {"Rosewood Lane", "Cassette Revival"}
unknown

That is the first trap pandas shares with SQL. (The `# noqa` comment above only
tells code checkers that the `== None` is deliberate.)

## 4. The five longest plays

In [ ]:
longest = plays.sort_values("minutes_played", ascending=False).head(5)
longest = longest[["track_name", "played_at", "minutes_played"]]

assert list(longest["track_name"]) == [
    "Paper Avenue", "Harbour Season", "Broken Lines", "Static Wires", "Northern Window",
]
longest

`sort_values(...).head(5)` is `ORDER BY ... DESC LIMIT 5`. There is also a
shortcut, `plays.nlargest(5, "minutes_played")`, which gives the same rows.

## 5. Plays in 2024 and in 2025

In [ ]:
per_year = plays["played_at"].str[:4].value_counts().sort_index()

assert per_year["2024"] == 1122
assert per_year["2025"] == 1061
per_year

`.str[:4]` slices every value in the column, the same as
`substr(played_at, 1, 4)` and the same as `date[:4]` on one string in
session 4. `.sort_index()` puts the years in order; without it,
`value_counts` would put the bigger count first.

## 6. Tracks played more than 40 times

In [ ]:
track_counts = plays["track_name"].value_counts()
popular = track_counts[track_counts > 40]

assert list(popular.index) == ["Broken Lines", "Salt Lines", "Amber Ferry", "Broken Mornings", "Open Orbit"]
assert list(popular) == [67, 63, 55, 52, 44]
popular

Count, then filter the counts: that is `HAVING COUNT(*) > 40`, with no
special keyword.

## 7. Total minutes per device, to the nearest minute

In [ ]:
device_minutes = plays.groupby("device")["minutes_played"].sum().round().astype(int)
device_minutes = device_minutes.sort_values(ascending=False)

assert device_minutes.to_dict() == {"phone": 4308, "laptop": 2144, "speaker": 1335, "car": 778, "tablet": 416}
device_minutes

`.round()` with nothing in the brackets rounds to whole numbers but leaves
them as decimals: `4308.0`. `.astype(int)` changes the type, so they print
as `4308`. SQL's `ROUND` does the same thing, and DB Browser hides it.

## 8. Genres with five or more artists

In [ ]:
genre_artists = artists["genre"].value_counts()
big_genres = genre_artists[genre_artists >= 5]

assert big_genres.to_dict() == {"Electronic": 7, "Indie": 6, "Folk": 5, "Jazz": 5, "Latin": 5, "Rock": 5}
big_genres

Notice the order of the four genres with 5. `value_counts` sorts by the
count, and among a tie the order is whatever it happens to be. In SQL you
added `genre` to the `ORDER BY` to settle it. The pandas equivalent sorts by
two things at once:

In [ ]:
tidy = big_genres.rename("artists").reset_index()
tidy.sort_values(["artists", "genre"], ascending=[False, True])

## 9. Skip rate per device

This is the trap SQL had and pandas does not.

In [ ]:
skips = plays.groupby("device").agg(
    plays=("play_id", "count"),
    skipped=("skipped", "sum"),
)
skips["skipped_pct"] = (100 * skips["skipped"] / skips["plays"]).round(1)
skips = skips.sort_values("skipped_pct", ascending=False)

assert skips["skipped_pct"].to_dict() == {"speaker": 14.0, "tablet": 13.3, "laptop": 11.5, "phone": 11.4, "car": 10.6}
skips

In SQLite, `19 / 179` was `0`, and you needed `100.0` to force a proper
division. In Python 3, `/` always gives the decimal answer, so the trap is
simply not there.

A new column is made by assigning to it: `skips["skipped_pct"] = ...`, the
way you added a key to a dictionary.

And a shortcut worth knowing: `skipped` holds only 0s and 1s, so its
**average** is the fraction of plays skipped. `mean` gives you the rate
directly:

In [ ]:
rate = (100 * plays.groupby("device")["skipped"].mean()).round(1)
assert rate.to_dict() == skips["skipped_pct"].to_dict()
rate.sort_values(ascending=False)

## 10. The busiest single day

In [ ]:
per_day = plays["played_at"].value_counts()

assert per_day.index[0] == "2024-03-24"
assert per_day.iloc[0] == 11
per_day.head(5)

2024-03-24 with 11 plays. And, as in session 6, look at second place before
believing it: 10, then three days on 8. A "busiest day" that wins by one play
out of 2,183 is not much of a finding.

`head(5)` instead of `head(1)` costs nothing, and shows you that. This is
the second trap pandas shares with SQL: no notation protects you from a
winner that is not really a winner.

# Task 2: an example

Task 2 is yours, but here is the first example from the session 6 answers,
translated. *Does listening get longer as the year goes on?* Plays and
average minutes per calendar month, both years together.

In [ ]:
plays["month"] = plays["played_at"].str[5:7]

by_month = plays.groupby("month").agg(
    plays=("play_id", "count"),
    avg_min=("minutes_played", "mean"),
).round(2)

assert by_month.loc["10", "plays"] == 239
assert set(by_month["plays"].nsmallest(3).index) == {"06", "07", "08"}
by_month

In [ ]:
print("plays per month, from", by_month["plays"].min(), "to", by_month["plays"].max())
print("average minutes, from", by_month["avg_min"].min(), "to", by_month["avg_min"].max())

The same answer as the SQL: the quiet months are June, July and August, but
the average play length barely moves all year. The seasonal effect is in
**how much** gets played, not how long each play lasts.

`plays["month"] = ...` adds a column to `plays` itself, made from slicing
another column. That is the pandas way of "the month is not a column, so
make one".

# Which was easier?

My answer, for comparison with yours:

* **Easier in pandas:** 5 (`.str[:4]` beats remembering `strftime`), 6 and
  8 (no `HAVING`), 9 (no integer division to trip over).
* **Easier in SQL:** 2, slightly. `WHERE country = 'Netherlands'` reads
  more like English than the double brackets do.
* **About the same:** the rest. Same ideas, different punctuation.

Neither language is better. They are two notations for one set of ideas,
and you now read both.